<a href="https://colab.research.google.com/github/BriennedeTarth/Random-Attractors-Found-using-Lyapunov-Exponents/blob/main/Random_Attractors_Found_using_Lyapunov_Exponents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title
# Python port of Paul Bourke's lyapunov/gen.c
# By Johan Bichel Lindegaard - http://johan.cc

import math
import random
from PIL import Image, ImageDraw
import argparse
import sys

parser = argparse.ArgumentParser(description='Search for chaos.')
# parser.add_argument('-i', dest='maxiterations' metavar='N', type=int,
#                     help='Maximum iterations.')

# Parse known arguments, ignoring the rest
args, unknown = parser.parse_known_args()

MAXITERATIONS = 100000
NEXAMPLES = 2000 # Increased number of examples

def createAttractor():
    for n in range(NEXAMPLES):
        lyapunov = 0
        xmin = 1e32
        xmax = -1e32
        ymin = 1e32
        ymax = -1e32
        ax, ay, x, y = [], [], [], []

        # Initialize coefficients for this attractor
        for i in range(6):
            # Slightly adjusted random range for coefficients
            ax.append(random.uniform(-2.5, 2.5))
            ay.append(random.uniform(-2.5, 2.5))

        # Calculate the attractor
        drawit = True
        x.append(random.uniform(-0.5, 0.5))
        y.append(random.uniform(-0.5, 0.5))

        d0 = -1
        while d0 <= 0:
            xe = x[0] + random.uniform(-0.5, 0.5) / 1000.0
            ye = y[0] + random.uniform(-0.5, 0.5) / 1000.0
            dx = x[0] - xe
            dy = y[0] - ye
            d0 = math.sqrt(dx * dx + dy * dy)

            for i in range(MAXITERATIONS):
                # Calculate next term
                x.append(
                    ax[0] + ax[1]*x[i-1] + ax[2]*x[i-1]*x[i-1] +
                    ax[3]*x[i-1]*y[i-1] + ax[4]*x[i-1]*x[i-1]*y[i-1] +
                    ax[5]*y[i-1]*y[i-1]
                )
                y.append(
                    ay[0] + ay[1]*y[i-1] + ay[2]*y[i-1]*y[i-1] +
                    ay[3]*y[i-1]*x[i-1] + ay[4]*y[i-1]*y[i-1]*x[i-1] +
                    ay[5]*x[i-1]*x[i-1]
                )

                # Calculate lyapunov exponent
                dx = x[i] - xe
                dy = y[i] - ye
                d1 = math.sqrt(dx * dx + dy * dy)
                lyapunov += math.log(d1 / d0)

                xe = x[i] + (xe - x[i]) * d0 / d1
                ye = y[i] + (ye - y[i]) * d0 / d1
                d0 = 1e-8

                # Check if the attractor is diverging
                if abs(x[i]) > 1e10 or abs(y[i]) > 1e10:
                    drawit = False
                    break

                # Check if the attractor is converging to a point
                if i > 100 and abs(x[i] - x[i-1]) < 1e-6 and abs(y[i] - y[i-1]) < 1e-6:
                    drawit = False
                    break

            # Calculate the lyapunov exponent
            if drawit:
                lyapunov /= MAXITERATIONS
                if lyapunov > 0:
                    print(f"Lyapunov exponent: {lyapunov}")

                    # Find bounding box
                    for i in range(MAXITERATIONS):
                        if i > 100:
                            xmin = min(xmin, x[i])
                            xmax = max(xmax, x[i])
                            ymin = min(ymin, y[i])
                            ymax = max(ymax, y[i])

                    # Draw the attractor
                    img = Image.new('RGB', (512, 512), color = 'black')
                    d = ImageDraw.Draw(img)
                    for i in range(MAXITERATIONS):
                        if i > 100:
                            ix = int((x[i] - xmin) / (xmax - xmin) * 511)
                            iy = int((y[i] - ymin) / (ymax - ymin) * 511)
                            if 0 <= ix < 512 and 0 <= iy < 512:
                                d.point((ix, iy), fill=(255, 255, 255))

                    img.save(f"attractor_{n}.png")

createAttractor()

Lyapunov exponent: 16.909677435503866
Lyapunov exponent: 17.247482511243728
Lyapunov exponent: 16.59586419067221
Lyapunov exponent: 18.157745353342072
